# Data Collection

In [ ]:
from bs4 import BeautifulSoup

# Sentimental Analysis via LSTM

In [10]:
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab_to_idx, max_length=50):
        self.texts = texts
        self.labels = labels
        self.vocab_to_idx = vocab_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        tokens = text.lower().split()
        token_ids = [self.vocab_to_idx.get(token, self.vocab_to_idx['<UNK>']) for token in tokens]

        if len(token_ids) < self.max_length:
            token_ids.extend([self.vocab_to_idx['<PAD>']] * (self.max_length - len(token_ids)))
        else:
            token_ids = token_ids[:self.max_length]

        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_classes, dropout=0.3):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        h0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.num_layers * 2, x.size(0), self.hidden_dim).to(x.device)

        lstm_out, _ = self.lstm(embedded, (h0, c0))

        output = self.dropout(lstm_out[:, -1, :])
        output = self.fc(output)

        return output

In [ ]:
# Preprocessing text

texts = []
labels = []

In [ ]:
# Split the data
X_temp, X_test, y_temp, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Create datasets and dataloaders
train_dataset = TextDataset(X_train, y_train, vocab_to_idx)
val_dataset = TextDataset(X_val, y_val, vocab_to_idx)
test_dataset = TextDataset(X_test, y_test, vocab_to_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Model Hyperparameters
vocab_size = len(vocab_to_idx)
embedding_dim = 128
hidden_dim = 64
num_layers = 2
num_classes = 2

# Initialize and train the model
model = LSTMClassifier(vocab_size, embedding_dim, hidden_dim, num_layers, num_classes, dropout=0.2)
model.to(device)

# The train_model function is in the notebook and handles the training loop.
# train_losses, val_losses, val_accuracies = train_model(model, train_loader, val_loader, num_epochs=20)
model.fit(train_loader, val_loader)

In [ ]:
test_accuracy, predictions, targets, probabilities = model.transform(test_loader)